## Data analysis & visualization: LAB 5 (code with answers) - "Clean the bike rentals"

In [ ]:
import pandas as pd

## Task 1:8 - Load it, and remove the empty columns

In [ ]:
rentals_raw = pd.read_csv('data/bike_rentals.csv')
rentals = rentals_raw.copy()

print(rentals.dtypes) # Before any cleaning of the data, 'price' has the str datatype.
print("\nShape:", rentals.shape)

In [ ]:
print(rentals.columns[rentals.isna().all()]) # Output the names of columns missing all values.

In [ ]:
rentals = rentals.dropna(axis=1, how='all') # Axis as in x=0, y=1. 1 means vertical (columns). Here we want to delete columns whereunder all values are missing.

In [ ]:
print(rentals.shape)

**Task 1 (^ above ^) - Answer: From 15 down to 12 after dropping (removing) columns with missing values. 0 columns left where values were missing.**
___

## Task 2:8 - The price: from text to numbers

In [ ]:
rentals['price'].head(10)

In [ ]:
rentals['price'] = rentals['price'].str.replace('SEK', '', regex=False)
rentals['price'] = rentals['price'].str.replace(' ', '', regex=False)
rentals['price'].head(3)

In [ ]:
rentals['price'] = rentals['price'].str.replace(',', '', regex=False)
rentals['price'].head(3)

In [ ]:
rentals['price'] = rentals['price'].astype(float)
print("Datatype of values in column 'price':", rentals['price'].dtype)
rentals['price'].describe()

In [ ]:
rentals[['rental_id', 'bike_type', 'status', 'price']].sort_values('price', ascending=False).head(3)
# rentals.nlargest(3, 'price')[['rental_id', 'bike_type', 'status', 'price']] # <<< Alternatively, this also works.

**Task 2 (^ above ^) - Answer: The steps above are self-explanatory, or, corresponds with the answers in at the end of Part 1.**
___

## Task 3:8 - The missing prices

In [ ]:
print(rentals['price'].isna().sum(), "rows for 'price' are missing.") # Sum of missing values.
print((rentals['price'].isna().mean() * 100).round(1), "% of 403 rows")

In [ ]:
price_missing = rentals['price'].isna()
rental_cancelled = rentals['status'] == 'Cancelled'

pd.crosstab(price_missing, rental_cancelled)

**Task 3 (^ above ^) - Answer: Of the 51 missing prices, 47 belong to cancelled rentals and have a clear reason. The other 4 are not cancelled and appear to be missing accidentally. I will leave all of them as missing rather than inventing prices.**
___

## Task 4:8 - The letters t and f

In [ ]:
# Using list comprehension to iterate over columns and their values.
tf_columns = [col for col in rentals.columns if set(rentals[col].dropna().unique()) <= {'t', 'f'}]

print(tf_columns) # 'status' contains three textual categories rather than the Boolean codes 't' and 'f', so it is correctly excluded.

In [ ]:
# Caution (!): If you run this more than once, you have to restart from the top of this notebook. Print statements in a separate note below.
rentals['is_member'] = rentals['is_member'].map({'t': True, 'f': False})
rentals['insured'] = rentals['insured'].map({'t': True, 'f': False})
rentals['paid_online'] = rentals['paid_online'].map({'t': True, 'f': False})

In [ ]:
# Here we can conclude that the insured column has 3 missing values (NaN).
print(rentals[['is_member', 'insured', 'paid_online']].isna().sum())
print(rentals['insured'].value_counts())
print(rentals[['is_member', 'insured', 'paid_online']].head(6))

In [ ]:
print(rentals['insured'].unique())
print(rentals['insured'].value_counts(dropna=False))

In [ ]:
rentals.groupby('is_member').agg(
    median_price=('price', 'median'),
    n_rentals=('price', 'count')
)

Task 5

start_date, end_date

In [ ]:
date_columns = ['start_date', 'end_date']

print(rentals[date_columns].dtypes)
rentals[date_columns].head(3)

In [ ]:
before_dt = rentals[date_columns].isna().sum()

for col in date_columns:
    rentals[col] = pd.to_datetime(rentals[col], errors='coerce')

after_dt = rentals[date_columns].isna().sum()

print(before_dt.equals(after_dt))
print(after_dt)
print(rentals[date_columns].dtypes)

**There were no missing date values before conversion. After conversion, start_date has two missing values because two strings could not be interpreted as valid dates. end_date still has no missing values.**

In [ ]:
rentals['rental_days'] = (
    rentals['end_date'] - rentals['start_date']
).dt.days

In [ ]:
rentals['rental_days'].describe()

In [ ]:
rentals[
    rentals['rental_days'] < 0
][
    ['rental_id', 'start_date', 'end_date', 'rental_days']
]